## Install and import

In [ ]:
!pip -q install scikit-multilearn iterative-stratification openpyxl

In [ ]:
import os, re, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score, jaccard_score,
                             hamming_loss, roc_auc_score, roc_curve, auc, average_precision_score,
                             precision_recall_curve, multilabel_confusion_matrix, classification_report)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
    try: keras.utils.set_random_seed(s)
    except Exception: pass
set_seed()

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## Configuration


In [ ]:
OPTIMIZATION_MODE = "quick"
USE_AUGMENTATION  = False
AUG_MASK_PROB, AUG_COPIES = 0.10, 1
THRESHOLD = 0.5
RUN_REPEATED_SEEDS = False
LABELS   = ["WhatsApp Chats", "SMS Messages", "Call Logs", "Contacts", "Media Files", "Browser History"]
FEATURES = ["Device Make", "Device Model", "OS Type", "OS Version", "Extraction Tool", "Tool Version", "Extraction Method"]


ALIASES = {
    "Device Make": ["Make", "Manufacturer", "Device Manufacturer", "Brand"],
    "Device Model": ["Model"],
    "OS Type": ["OS", "Operating System", "OS Name"],
    "OS Version": ["Android Version", "OS Ver"],
    "Extraction Tool": ["Tool", "Forensic Tool", "Acquisition Tool"],
    "Tool Version": ["Tool Ver", "Version of Tool"],
    "Extraction Method": ["Method", "Acquisition Method", "Extraction Type"],
    "WhatsApp Chats": ["WhatsApp", "Whatsapp Chat"],
    "SMS Messages": ["SMS", "Messages"],
    "Call Logs": ["Call Log", "Calls"],
    "Contacts": ["Contact"],
    "Media Files": ["Media", "Multimedia"],
    "Browser History": ["Browser", "Browsing History"],
}

TEXT_FIELD_ALIASES = ["Recovered Artifacts", "Recovered Artefacts", "Artifacts Recovered"]
KEYWORDS = {
    "WhatsApp Chats":  r"whats\s*app",
    "SMS Messages":    r"\bsms\b|text message",
    "Call Logs":       r"call\s*log|call\s*history|\bcalls?\b",
    "Contacts":        r"contact",
    "Media Files":     r"media|image|photo|picture|video|audio|multimedia",
    "Browser History": r"browser|browsing|web\s*history",
}


PAPER_PARAMS = dict(emb=32, heads=4, layers=2, dff=128, kan=64, dropout=0.30,
                    lr=1e-4, batch=16, epochs=100, patience=15)

In [ ]:
def upload_dataset():
    try:                                              # Google Colab
        from google.colab import files
        up = files.upload()
        return next(iter(up))
    except ImportError:
        pass
    try:                                              # local Jupyter with a display
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        path = filedialog.askopenfilename(title="Select the forensic dataset",
                                          filetypes=[("Data files", "*.csv *.xlsx *.xls"), ("All files", "*.*")])
        root.destroy()
        if path: return path
    except Exception:
        pass
    return input("Enter the full path to the dataset file: ").strip().strip('"')

DATA_PATH = upload_dataset()
print("Selected:", DATA_PATH)

Saving archive (2) (1).zip to archive (2) (1).zip
Selected: archive (2) (1).zip


In [ ]:
def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())

def find_col(name, df, extra=()):
    lookup = {norm(c): c for c in df.columns}
    for cand in [name, *ALIASES.get(name, []), *extra]:
        if norm(cand) in lookup: return lookup[norm(cand)]
    return None

feat_cols = {f: find_col(f, raw) for f in FEATURES}
lab_cols  = {l: find_col(l, raw) for l in LABELS}
print("Feature columns :", feat_cols)
print("Label columns   :", lab_cols)
missing_f = [f for f, c in feat_cols.items() if c is None]
assert not missing_f, f"Feature columns not found: {missing_f}. Add spellings to ALIASES."

POS = {"1", "1.0", "yes", "y", "true", "t", "recovered", "available", "present", "✓", "✔"}
def to_bin(v):
    if pd.isna(v): return 0
    return int(str(v).strip().lower() in POS)

df = raw.copy()
if all(c is not None for c in lab_cols.values()):
    for l, c in lab_cols.items(): df[l] = raw[c].map(to_bin)
    print("Targets taken from the six label columns.")
else:
    txt = find_col(TEXT_FIELD_ALIASES[0], raw, extra=TEXT_FIELD_ALIASES[1:])
    assert txt is not None, "Neither the six label columns nor a 'Recovered Artifacts' text column was found."
    print(f"Targets parsed from the free-text column '{txt}'")
    s = raw[txt].fillna("").astype(str).str.lower()
    for l in LABELS: df[l] = s.str.contains(KEYWORDS[l], regex=True).astype(int)

df = df.drop_duplicates().reset_index(drop=True)                       # exact duplicate records only
print("After removing exact duplicates:", df.shape)


Feature columns : {'Device Make': 'Device Make', 'Device Model': 'Model', 'OS Type': 'OS Type', 'OS Version': 'OS Version', 'Extraction Tool': 'Extraction Tool', 'Tool Version': 'Tool Version', 'Extraction Method': 'Extraction Method'}
Label columns   : {'WhatsApp Chats': 'WhatsApp Chats', 'SMS Messages': 'SMS Messages', 'Call Logs': 'Call Logs', 'Contacts': 'Contacts', 'Media Files': 'Media Files', 'Browser History': 'Browser History'}
Targets taken from the six label columns.
After removing exact duplicates: (1500, 17)


## Pre-processing, iterative stratified split, optional augmentation


In [ ]:
PLACEHOLDERS = {"", "nan", "none", "-", "n/a", "na", "null", "unknown", "not available"}

def clean_feature(s):
    s = s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    s = s.where(~s.str.lower().isin(PLACEHOLDERS), "Unknown")
    canon = {}                                                       #
    for v in s: canon.setdefault(v.lower(), v)
    return s.str.lower().map(canon)

# Map each feature using its matched column name from feat_cols
X_df = pd.DataFrame({f: clean_feature(df[feat_cols[f]]) for f in FEATURES})
Y = df[LABELS].values.astype("float32")
n = len(X_df)

def stratified_split(Y, n_test, n_val, seed=SEED):
    idx = np.arange(len(Y)).reshape(-1, 1)
    try:
        from skmultilearn.model_selection import iterative_train_test_split
        np.random.seed(seed)
        i_tmp, y_tmp, i_te, y_te = iterative_train_test_split(idx, Y, test_size=n_test / len(Y))
        i_tr, y_tr, i_va, y_va = iterative_train_test_split(i_tmp, y_tmp, test_size=n_val / len(i_tmp))
        return i_tr.ravel(), i_va.ravel(), i_te.ravel(), "skmultilearn iterative stratification"
    except Exception as e1:
        try:
            from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
            s1 = MultilabelStratifiedShuffleSplit(1, test_size=n_test / len(Y), random_state=seed)
            tmp, te = next(s1.split(idx, Y))
            s2 = MultilabelStratifiedShuffleSplit(1, test_size=n_val / len(tmp), random_state=seed)
            tr_i, va_i = next(s2.split(idx[tmp], Y[tmp]))
            return tmp[tr_i], tmp[va_i], te, "iterative-stratification (MultilabelStratifiedShuffleSplit)"
        except Exception as e2:
            print("WARNING: no multilabel-stratification package available — falling back to a RANDOM split.")
            perm = np.random.default_rng(seed).permutation(len(Y))
            return perm[n_test + n_val:], perm[n_test:n_test + n_val], perm[:n_test], "random split (fallback)"

N_TEST = N_VAL = int(round(0.1507 * n))
tr_idx, va_idx, te_idx, split_name = stratified_split(Y, N_TEST, N_VAL)
print(f"Split method: {split_name}")
print(f"Train {len(tr_idx)} ({len(tr_idx)/n:.2%}) | Val {len(va_idx)} ({len(va_idx)/n:.2%}) | Test {len(te_idx)} ({len(te_idx)/n:.2%})")

# vocabularies from the training split only; index 0 = unknown
vocab = {f: {c: i + 1 for i, c in enumerate(sorted(X_df.loc[tr_idx, f].unique()))} for f in FEATURES}
cards = [len(vocab[f]) for f in FEATURES]
def encode(idx): return np.stack([X_df.loc[idx, f].map(vocab[f]).fillna(0).astype("int32").values for f in FEATURES], 1)

X_tr, X_va, X_te = encode(tr_idx), encode(va_idx), encode(te_idx)
y_tr, y_va, y_te = Y[tr_idx], Y[va_idx], Y[te_idx]
print("Cardinalities:", dict(zip(FEATURES, cards)))
print(f"Unseen-category rate  val: {(X_va==0).mean():.2%} | test: {(X_te==0).mean():.2%}")
share = pd.DataFrame({"Train": y_tr.mean(0), "Val": y_va.mean(0), "Test": y_te.mean(0)}, index=LABELS).mul(100).round(1)
print("\nPositive-label rate (%) per split:"); display(share)

Split method: skmultilearn iterative stratification
Train 1048 (69.87%) | Val 226 (15.07%) | Test 226 (15.07%)
Cardinalities: {'Device Make': 10, 'Device Model': 27, 'OS Type': 3, 'OS Version': 9, 'Extraction Tool': 5, 'Tool Version': 5, 'Extraction Method': 5}
Unseen-category rate  val: 0.00% | test: 0.00%

Positive-label rate (%) per split:


,Train,Val,Test
WhatsApp Chats,58.400002,55.299999,62.400002
SMS Messages,80.500000,80.099998,77.900002
Call Logs,87.199997,86.699997,87.199997
Contacts,93.000000,92.000000,92.900002
Media Files,69.099998,68.599998,67.699997
Browser History,52.200001,51.799999,54.000000


In [ ]:
# Optional augmentation (training split only). Disabled by default
def augment_by_masking(X, y, p=0.10, copies=1, seed=SEED):
    rng = np.random.default_rng(seed); Xs, ys = [X], [y]
    for _ in range(copies):
        Xa = X.copy(); Xa[rng.random(X.shape) < p] = 0
        Xs.append(Xa); ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

if USE_AUGMENTATION:
    X_tr_aug, y_tr_aug = augment_by_masking(X_tr, y_tr, AUG_MASK_PROB, AUG_COPIES)
    print(f"Augmentation ON: training set {len(X_tr)} → {len(X_tr_aug)} records")
else:
    X_tr_aug, y_tr_aug = X_tr, y_tr
    print("Augmentation OFF .")

Augmentation OFF .


## Model: categorical embeddings → TabTransformer → KAN → sigmoid head

In [ ]:
class FeatureEmbedding(layers.Layer):
    """One embedding table per categorical attribute (+ learnable column embedding)."""
    def __init__(self, cards, dim, **kw):
        super().__init__(**kw)
        self.cards, self.dim = list(cards), dim
        self.offsets = np.cumsum([0] + [c + 1 for c in self.cards[:-1]]).astype("int32")
        self.emb = layers.Embedding(sum(c + 1 for c in self.cards), dim)
        self.col = layers.Embedding(len(self.cards), dim)
    def call(self, x):
        e = self.emb(tf.cast(x, tf.int32) + tf.constant(self.offsets))          # (B, p, d)
        return e + self.col(tf.range(len(self.cards)))[None]
    def get_config(self):
        return {**super().get_config(), "cards": self.cards, "dim": self.dim}

class KANLayer(layers.Layer):
    """phi(x) = w_b*SiLU(x) + w_s * sum_r c_r B_r(x)  with cubic B-splines, G=5 on [-2,2] (R = G+k = 8)."""
    def __init__(self, units, grid_size=5, spline_order=3, grid_range=(-2.0, 2.0), l1=1e-4, **kw):
        super().__init__(**kw)
        self.units, self.G, self.k, self.range, self.l1 = units, grid_size, spline_order, grid_range, l1
        h = (grid_range[1] - grid_range[0]) / grid_size
        self.grid = (np.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]).astype("float32")
        self.ln = layers.LayerNormalization()
    def build(self, input_shape):
        d, R = int(input_shape[-1]), self.G + self.k
        self.ln.build(input_shape)
        self.base_w = self.add_weight(name="base_w", shape=(d, self.units), initializer="glorot_uniform")
        self.spline_w = self.add_weight(name="spline_w", shape=(d, self.units, R),
                                        initializer=keras.initializers.RandomNormal(0.0, 0.1),
                                        regularizer=keras.regularizers.L1(self.l1))
        self.spline_s = self.add_weight(name="spline_s", shape=(d, self.units), initializer="ones")
        super().build(input_shape)
    def b_splines(self, x):                                                     # Cox–de Boor recursion
        g = tf.constant(self.grid); x = tf.expand_dims(x, -1)
        B = tf.cast((x >= g[:-1]) & (x < g[1:]), x.dtype)
        for p in range(1, self.k + 1):
            B = ((x - g[:-(p + 1)]) / (g[p:-1] - g[:-(p + 1)])) * B[..., :-1] + \
                ((g[p + 1:] - x) / (g[p + 1:] - g[1:-p])) * B[..., 1:]
        return B                                                                # (B, d, G+k)
    def call(self, x):
        x = self.ln(x)
        base = tf.matmul(tf.nn.silu(x), self.base_w)
        spline = tf.einsum("bir,ior->bo", self.b_splines(x), self.spline_w * self.spline_s[..., None])
        return base + spline
    def get_config(self):
        return {**super().get_config(), "units": self.units, "grid_size": self.G, "spline_order": self.k,
                "grid_range": self.range, "l1": self.l1}

def build_model(p, cards, n_labels=len(LABELS)):
    d, H = int(p["emb"]), int(p["heads"])
    assert d % H == 0, "embedding dim must be divisible by #heads"
    inp = keras.Input(shape=(len(cards),), dtype="int32", name="categorical_inputs")
    x = FeatureEmbedding(cards, d)(inp)
    for _ in range(int(p["layers"])):                                           # TabTransformer encoder blocks
        a = layers.MultiHeadAttention(num_heads=H, key_dim=d // H, dropout=p["dropout"])(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(a))
        f = layers.Dense(int(p["dff"]), activation="gelu")(x)
        f = layers.Dense(d)(layers.Dropout(p["dropout"])(f))
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(f))
    zt = layers.Flatten()(x)                                                    # Z_T  (7 × d)
    zk = KANLayer(int(p["kan"]))(zt)                                            # KAN: (7d) → d_KAN → d_KAN
    zk = layers.Dropout(p["dropout"])(zk)
    zk = KANLayer(int(p["kan"]))(zk)                                            # Z_K
    h = layers.Concatenate()([zt, zk])                                          # h = [Z_T ; Z_K]
    h = layers.Dropout(p["dropout"])(h)
    out = layers.Dense(n_labels, activation="sigmoid", name="artifact_probs")(h)
    model = keras.Model(inp, out, name="ForensicTAB")
    try:    opt = keras.optimizers.AdamW(learning_rate=p["lr"], weight_decay=1e-5)
    except AttributeError: opt = keras.optimizers.Adam(learning_rate=p["lr"])
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
    return model

m = build_model(PAPER_PARAMS, cards); m.summary(); del m

Model: "ForensicTAB"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ categorical_inputs  │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_embedding   │ (None, 7, 32)     │      2,496 │ categorical_inpu… │
│ (FeatureEmbedding)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 32)     │      4,224 │ feature_embeddin… │
│ (MultiHeadAttentio… │                   │            │ feature_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 7, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 7, 32)     │          0 │ feature_embeddin… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 7, 32)     │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 7, 128)    │      4,224 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 7, 128)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 7, 32)     │      4,128 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 7, 32)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 7, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 32)     │      4,224 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 7, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 7, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 32)     │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 7, 128)    │      4,224 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 7, 128)    │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 7, 32)     │      4,128 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 7, 32)     │          0 │ dense_3[0][0]   

 Total params: 214,534 (838.02 KB)

 Trainable params: 214,534 (838.02 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def train_model(p, Xtr, ytr, Xva, yva, seed=SEED, epochs=None, verbose=0):
    keras.backend.clear_session(); set_seed(seed)
    model = build_model(p, cards)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=int(p["patience"]), restore_best_weights=True)
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=int(epochs or p["epochs"]),
                     batch_size=int(p["batch"]), callbacks=[es], verbose=verbose, shuffle=True)
    return model, hist

def macro_f1(model, X, y, thr=THRESHOLD):
    return f1_score(y, (model.predict(X, verbose=0) >= thr).astype(int), average="macro", zero_division=0)

##  Hyper-parameter optimisation with the Dung Beetle Optimizer


In [ ]:
SPACE = [("emb",      "choice", [16, 32, 48, 64]),
         ("heads",    "choice", [2, 4, 8]),
         ("layers",   "choice", [1, 2, 3, 4]),
         ("dff",      "choice", [64, 128, 256]),
         ("kan",      "choice", [32, 64, 128]),
         ("dropout",  "float",  (0.10, 0.50)),
         ("lr",       "log",    (1e-5, 1e-3)),
         ("batch",    "choice", [16, 32, 64]),
         ("epochs",   "int",    (50, 150)),
         ("patience", "int",    (10, 20))]

def decode(v):
    p = {}
    for u, (name, kind, rng) in zip(np.clip(v, 0, 1), SPACE):
        if kind == "choice": p[name] = rng[int(round(u * (len(rng) - 1)))]
        elif kind == "float": p[name] = float(rng[0] + u * (rng[1] - rng[0]))
        elif kind == "log":   p[name] = float(10 ** (np.log10(rng[0]) + u * (np.log10(rng[1]) - np.log10(rng[0]))))
        else:                 p[name] = int(round(rng[0] + u * (rng[1] - rng[0])))
    return p

def dbo(fitness_fn, dim, N=20, T=30, seed=SEED, log=print):
    rng = np.random.default_rng(seed)
    n_roll = max(1, round(N * 4 / 20)); n_brood = max(1, round(N * 4 / 20)); n_small = max(1, round(N * 5 / 20))
    k, b, S = 0.1, 0.3, 0.5
    pop = rng.random((N, dim)); fit = np.array([fitness_fn(x) for x in pop]); prev = pop.copy()
    gi = int(np.argmin(fit)); gbest, gfit = pop[gi].copy(), fit[gi]
    history = [1 - gfit]; log(f"[DBO] init  best val macro-F1 = {1-gfit:.4f}")
    for t in range(1, T + 1):
        worst, loc = pop[np.argmax(fit)].copy(), pop[np.argmin(fit)].copy()
        R = 1 - t / T
        lb_g, ub_g = np.maximum(gbest * (1 - R), 0), np.minimum(gbest * (1 + R), 1)
        lb_l, ub_l = np.maximum(loc * (1 - R), 0),   np.minimum(loc * (1 + R), 1)
        new = pop.copy()
        for i in range(N):
            if i < n_roll:
                if rng.random() < 0.9:
                    alpha = 1 if rng.random() < 0.9 else -1
                    new[i] = pop[i] + alpha * k * prev[i] + b * np.abs(pop[i] - worst)
                else:
                    psi = rng.uniform(0, np.pi)
                    if not (np.isclose(psi, 0) or np.isclose(psi, np.pi / 2) or np.isclose(psi, np.pi)):
                        new[i] = pop[i] + np.tan(psi) * np.abs(pop[i] - prev[i])
            elif i < n_roll + n_brood:                                         # brood balls
                new[i] = loc + rng.normal(size=dim) * (pop[i] - lb_l) + rng.random(dim) * (pop[i] - ub_l)
            elif i < n_roll + n_brood + n_small:                               # small beetles
                new[i] = pop[i] + rng.normal(size=dim) * (pop[i] - lb_g) + rng.random(dim) * (pop[i] - ub_g)
            else:                                                              # thieves
                new[i] = gbest + S * rng.normal(size=dim) * (np.abs(pop[i] - loc) + np.abs(pop[i] - gbest))
        new = np.clip(new, 0, 1)
        new_fit = np.array([fitness_fn(x) for x in new])
        better = new_fit < fit
        prev = pop.copy(); pop[better], fit[better] = new[better], new_fit[better]
        gi = int(np.argmin(fit))
        if fit[gi] < gfit: gbest, gfit = pop[gi].copy(), fit[gi]
        history.append(1 - gfit); log(f"[DBO] iter {t:>2}/{T}  best val macro-F1 = {1-gfit:.4f}")
    return gbest, gfit, history

In [ ]:
_cache = {}
def make_fitness(epoch_cap=None):
    def fitness(v):
        p = decode(v); key = json.dumps(p, sort_keys=True) + str(epoch_cap)
        if key not in _cache:
            try:
                model, _ = train_model(p, X_tr_aug, y_tr_aug, X_va, y_va, seed=SEED, epochs=epoch_cap or p["epochs"])
                _cache[key] = 1 - macro_f1(model, X_va, y_va)
            except Exception:
                _cache[key] = 1.0
        return _cache[key]
    return fitness

best_params, dbo_history = dict(PAPER_PARAMS), None
if OPTIMIZATION_MODE != "skip":
    N_POP, N_ITER, CAP = (20, 30, None) if OPTIMIZATION_MODE == "full" else (8, 5, 40)
    gbest, gfit, dbo_history = dbo(make_fitness(CAP), len(SPACE), N_POP, N_ITER)
    best_params = decode(gbest)

json.dump(best_params, open(f"{OUT_DIR}/best_hyperparameters.json", "w"), indent=2)
print("Selected hyper-parameters:")
print("Hyperparameter  Value")
for k, v in best_params.items():
    v = f"{v:.4g}" if isinstance(v, float) else str(v)
    print(f"{k:>14}  {v:>5}")

Selected hyper-parameters:
Hyperparameter  Value
           emb     32
         heads      4
        layers      2
           dff    128
           kan     64
       dropout   0.25
            lr 0.0008
         batch     64
        epochs    120
      patience     15


## Final training with the selected configuration

In [ ]:
model, history = train_model(best_params, X_tr_aug, y_tr_aug, X_va, y_va, seed=SEED, verbose=2)
hist = pd.DataFrame(history.history); hist.to_csv(f"{OUT_DIR}/training_history.csv", index=False)


## Results on the independent test set

In [ ]:
prob_te = model.predict(X_te, verbose=0)
pred_te = (prob_te >= THRESHOLD).astype(int)
yt = y_te.astype(int)


# Repeated-seed analysis and controlled ablation



### Cell 1: settings, helpers, generic model builder

In [ ]:
import json, os, re, time
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             jaccard_score, hamming_loss)



FINAL_SEEDS = list(range(10))
ABL_TUNING  = "quick"
SEED_DIR    = f"{OUT_DIR}/seed_runs"; os.makedirs(SEED_DIR, exist_ok=True)
TUNED_JSON  = f"{OUT_DIR}/ablation_tuned_params.json"


DEFAULT_PARAMS = dict(emb=48, heads=4, layers=2, dff=128, kan=64, dropout=0.30,
                      lr=3e-4, batch=32, epochs=100, patience=15)

def pct(x): return round(100 * x, 2)

def full_metrics(y, prob, thr=THRESHOLD):
    y = np.asarray(y).astype(int); pred = (prob >= thr).astype(int)
    return {"Accuracy (%)":        pct((y == pred).mean()),
            "Macro precision (%)": pct(precision_score(y, pred, average="macro", zero_division=0)),
            "Macro recall (%)":    pct(recall_score(y, pred, average="macro", zero_division=0)),
            "Macro F1 (%)":        pct(f1_score(y, pred, average="macro", zero_division=0)),
            "Jaccard (%)":         pct(jaccard_score(y, pred, average="samples", zero_division=0)),
            "Hamming loss (%)":    pct(hamming_loss(y, pred)),
            "Exact match (%)":     pct(accuracy_score(y, pred))}

METRIC_COLS = ["Accuracy (%)", "Macro precision (%)", "Macro recall (%)", "Macro F1 (%)",
               "Jaccard (%)", "Hamming loss (%)", "Exact match (%)"]

def n_trainable(model): return int(sum(np.prod(w.shape) for w in model.trainable_weights))

def build_variant(p, cards, variant, mlp_units=None, n_labels=len(LABELS)):
    """variant: 'hybrid' (= your build_model) | 'tt_only' | 'tt_mlp' | 'kan_only'"""
    if variant == "hybrid":
        return build_model(p, cards, n_labels)
    d, H = int(p["emb"]), int(p["heads"])
    inp = keras.Input(shape=(len(cards),), dtype="int32", name="categorical_inputs")
    x = FeatureEmbedding(cards, d)(inp)
    n_enc = 0 if variant == "kan_only" else int(p["layers"])
    for _ in range(n_enc):
        a = layers.MultiHeadAttention(num_heads=H, key_dim=d // H, dropout=p["dropout"])(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(a))
        f = layers.Dense(int(p["dff"]), activation="gelu")(x)
        f = layers.Dense(d)(layers.Dropout(p["dropout"])(f))
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(f))
    zt = layers.Flatten()(x)
    if variant == "tt_only":
        h = zt
    elif variant == "kan_only":
        zk = KANLayer(int(p["kan"]))(zt); zk = layers.Dropout(p["dropout"])(zk); zk = KANLayer(int(p["kan"]))(zk)
        h = layers.Concatenate()([zt, zk])
    elif variant == "tt_mlp":
        zm = layers.Dense(int(mlp_units), activation="gelu")(zt); zm = layers.Dropout(p["dropout"])(zm)
        zm = layers.Dense(int(mlp_units), activation="gelu")(zm)
        h = layers.Concatenate()([zt, zm])
    else:
        raise ValueError(variant)
    h = layers.Dropout(p["dropout"])(h)
    out = layers.Dense(n_labels, activation="sigmoid", name="artifact_probs")(h)
    model = keras.Model(inp, out, name=f"ForensicTAB_{variant}")
    try:    opt = keras.optimizers.AdamW(learning_rate=p["lr"], weight_decay=1e-5)
    except AttributeError: opt = keras.optimizers.Adam(learning_rate=p["lr"])
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
    return model

def train_variant(p, variant, seed, mlp_units=None, epochs=None, verbose=0):
    """Same protocol as train_model: val_loss early stopping, best weights restored, same data/split."""
    keras.backend.clear_session(); set_seed(seed)
    model = build_variant(p, cards, variant, mlp_units)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=int(p["patience"]), restore_best_weights=True)
    hist = model.fit(X_tr_aug, y_tr_aug, validation_data=(X_va, y_va), epochs=int(epochs or p["epochs"]),
                     batch_size=int(p["batch"]), callbacks=[es], verbose=verbose, shuffle=True)
    return model, hist

### repeated seeds for the proposed model

In [ ]:
RESULTS = {}
PROPOSED = "TabTransformer + KAN hybrid, DBO-tuned (proposed)"

def run_seeds(name, p, variant, mlp_units=None, seeds=FINAL_SEEDS):
    if name in RESULTS: return RESULTS[name]
    rows, probs = [], []
    for s in seeds:
        t0 = time.time()
        model, hist = train_variant(p, variant, s, mlp_units)
        prob = model.predict(X_te, verbose=0)
        m = full_metrics(y_te, prob)
        m.update(seed=s, epochs_run=len(hist.history["loss"]),
                 best_val_loss=round(float(min(hist.history["val_loss"])), 4), params=n_trainable(model))
        rows.append(m); probs.append(prob)
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]
        np.save(f"{SEED_DIR}/{slug}_seed{s}_test_probs.npy", prob.astype("float32"))
    RESULTS[name] = dict(df=pd.DataFrame(rows), probs=probs, params=p, variant=variant, mlp_units=mlp_units)
    return RESULTS[name]

def summarize(df, cols=METRIC_COLS):
    n = len(df); tc = stats.t.ppf(0.975, n - 1); out = {}
    for c in cols:
        mu, sd = df[c].mean(), df[c].std(ddof=1); h = tc * sd / np.sqrt(n)
        out[c] = dict(mean=mu, sd=sd, ci_low=mu - h, ci_high=mu + h)
    return pd.DataFrame(out).T

res = run_seeds(PROPOSED, best_params, "hybrid")
res["df"].to_csv(f"{OUT_DIR}/repeated_seed_runs.csv", index=False)
tab10a = summarize(res["df"]).round(2)
tab10a["mean ± SD"] = tab10a.apply(lambda r: f"{r['mean']:.2f} ± {r['sd']:.2f}", axis=1)
tab10a["95% t-interval"] = tab10a.apply(lambda r: f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]", axis=1)
tab10a.to_csv(f"{OUT_DIR}/table_ten_seeds.csv")
print("\n=== Ten-seed summary of the proposed model (seeds 0-9) ===")
print(tab10a[["mean ± SD", "95% t-interval"]].to_string())

In [ ]:
import re

NICE = {"accuracy": "accuracy (%)", "macroprecision": "macro-precision (%)", "macrorecall": "macro-recall (%)",
        "macrof1": "macro-F1 (%)", "jaccard": "Jaccard (%)", "exactmatch": "exact match (%)"}

def nice(c):
    k = re.sub(r"[^a-z0-9]", "", c.lower().replace("(%)", ""))
    return NICE.get(k, c)

In [5]:
RESULTS = {}
PROPOSED = "TabTransformer + KAN hybrid, DBO-tuned (proposed)"

def run_seeds(name, p, variant, mlp_units=None, seeds=FINAL_SEEDS):
    if name in RESULTS: return RESULTS[name]
    rows, probs = [], []
    for s in seeds:
        t0 = time.time()
        model, hist = train_variant(p, variant, s, mlp_units)
        prob = model.predict(X_te, verbose=0)
        m = full_metrics(y_te, prob)
        m.update(seed=s, epochs_run=len(hist.history["loss"]),
                 best_val_loss=round(float(min(hist.history["val_loss"])), 4), params=n_trainable(model))
        rows.append(m); probs.append(prob)
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]
        np.save(f"{SEED_DIR}/{slug}_seed{s}_test_probs.npy", prob.astype("float32"))
    RESULTS[name] = dict(df=pd.DataFrame(rows), probs=probs, params=p, variant=variant, mlp_units=mlp_units)
    return RESULTS[name]

def summarize(df, cols=METRIC_COLS):
    n = len(df); tc = stats.t.ppf(0.975, n - 1); out = {}
    for c in cols:
        mu, sd = df[c].mean(), df[c].std(ddof=1); h = tc * sd / np.sqrt(n)
        out[c] = dict(mean=mu, sd=sd, ci_low=mu - h, ci_high=mu + h)
    return pd.DataFrame(out).T

res = run_seeds(PROPOSED, best_params, "hybrid")
res["df"].to_csv(f"{OUT_DIR}/repeated_seed_runs.csv", index=False)
tab10a = summarize(res["df"]).round(2)
tab10a["mean ± SD"] = tab10a.apply(lambda r: f"{r['mean']:.2f} ± {r['sd']:.2f}", axis=1)
tab10a["95% t-interval"] = tab10a.apply(lambda r: f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]", axis=1)
tab10a.to_csv(f"{OUT_DIR}/table_ten_seeds.csv")
ICE = {"accuracy": "accuracy (%)", "macroprecision": "macro-precision (%)", "macrorecall": "macro-recall (%)",
        "macrof1": "macro-F1 (%)", "jaccard": "Jaccard (%)", "exactmatch": "exact match (%)"}
def nice(c):
    k = re.sub(r"[^a-z0-9]", "", c.lower().replace("(%)", ""))
    return NICE.get(k, c)
tab10a_fmt = pd.DataFrame({
    "Quantity":     ["Ten seeds: " + nice(c) for c in tab10a.index],
    "Estimate":     [f"{r['mean']:.2f} ± {r['sd']:.2f}" for _, r in tab10a.iterrows()],
    "95% interval": [f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]" for _, r in tab10a.iterrows()],
    "Holm-adj. p":  "—"})
tab10a_fmt.to_csv(f"{OUT_DIR}/table_ten_seeds_formatted.csv", index=False)
print("\n=== Ten-seed summary of the proposed model (seeds 0-9) ===")
print(tab10a_fmt.to_string(index=False))

== Ten-seed summary of the proposed model (seeds 0-9) ===
                       Quantity     Estimate   95% interval Holm-adj. p
        Ten seeds: accuracy (%) 98.21 ± 0.22 [98.05, 98.37]           —
 Ten seeds: macro-precision (%) 98.84 ± 0.25 [98.66, 99.02]           —
    Ten seeds: macro-recall (%) 98.45 ± 0.27 [98.26, 98.64]           —
        Ten seeds: macro-F1 (%) 98.62 ± 0.24 [98.45, 98.79]           —
         Ten seeds: Jaccard (%) 97.38 ± 0.34 [97.14, 97.62]           —
     Ten seeds: exact match (%) 92.69 ± 0.88 [92.06, 93.32]           —


### bootstrap on the reported run

In [7]:
def fast_macro_f1(y, pred):
    tp = (y & pred).sum(0); fp = ((1 - y) & pred).sum(0); fn = (y & (1 - pred)).sum(0)
    den = 2 * tp + fp + fn
    return float(np.mean(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)))
def _metrics_fast(y, pred):
    return dict(acc=(y == pred).mean(), f1=fast_macro_f1(y, pred),
                jac=np.mean((y & pred).sum(1) / np.maximum((y | pred).sum(1), 1)),
                exact=(y == pred).all(1).mean())

def bootstrap_single_run(prob, y=y_te, B=2000, seed=SEED, thr=THRESHOLD):
    y = np.asarray(y).astype(int); pred = (prob >= thr).astype(int)
    rng = np.random.default_rng(seed); n = len(y); draws = []
    for _ in range(B):
        i = rng.integers(0, n, n); m = _metrics_fast(y[i], pred[i]); draws.append([m["acc"], m["f1"], m["jac"], m["exact"]])
    draws = 100 * np.array(draws); pt = _metrics_fast(y, pred)
    rows = []
    for k, name in enumerate(["Accuracy (%)", "Macro-F1 (%)", "Jaccard (%)", "Exact match (%)"]):
        lo, hi = np.percentile(draws[:, k], [2.5, 97.5]); rows.append((name, round(100 * [pt["acc"], pt["f1"], pt["jac"], pt["exact"]][k], 2), round(lo, 2), round(hi, 2)))
    return pd.DataFrame(rows, columns=["Metric", "Point estimate", "95% CI low", "95% CI high"])

tab10b = bootstrap_single_run(prob_te)
tab10b.to_csv(f"{OUT_DIR}/table10B_bootstrap.csv", index=False)

LABEL = {"Accuracy (%)": "accuracy (%)", "Macro-F1 (%)": "macro-F1 (%)",
         "Jaccard (%)": "Jaccard (%)", "Exact match (%)": "exact match (%)"}
tab10b_fmt = pd.DataFrame({
    "Quantity":     ["B. Bootstrap, reported run: " + LABEL[m] for m in tab10b["Metric"]],
    "Estimate":     tab10b["Point estimate"].map("{:.2f}".format),
    "95% interval": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(tab10b["95% CI low"], tab10b["95% CI high"])],
    "Holm-adj. p":  "—"})
tab10b_fmt.to_csv(f"{OUT_DIR}/table10B_bootstrap_formatted.csv", index=False)
print("\n=== Bootstrap (2,000 resamples) of the reported run ===")
print(tab10b_fmt.to_string(index=False))

== Bootstrap (2,000 resamples) of the reported run ===
                                Quantity Estimate   95% interval Holm-adj. p
   Bootstrap, reported run: accuracy (%)    98.38 [97.64, 98.97]           —
   Bootstrap, reported run: macro-F1 (%)    98.81 [97.93, 99.51]           —
    Bootstrap, reported run: Jaccard (%)    97.65 [96.42, 98.62]           —
Bootstrap, reported run: exact match (%)    93.36 [89.82, 96.46]           —
